# Two-Stage Early-Prediction Model — Retention & Activeness

Reproduces the two-stage model from the chapter:
- **Stage 1 (retention):** among all first-observed-week users, predict who churns from the feed.
- **Stage 2 (activeness):** among returning users, predict who is inactive (zero clicks).

**Prerequisites** (run from repo root first, if not done):
```bash
bash src/data/run_user_window_agg.sh "../Dataset/Raw_Data.zip"
python -m src.data.build_labels
bash src/data/run_content_taste.sh "../Dataset/Raw_Data.zip"   # also extracts user_demographics
python -m src.data.build_retention
```
Install plotting/ML deps: `pip3 install matplotlib scikit-learn jupyter`.


## Setup


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import polars as pl, numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve
from sklearn.inspection import permutation_importance
from src import config

DER = config.DERIVED_DIR
plt.rcParams.update({"figure.figsize": (6, 4), "axes.grid": True, "grid.alpha": 0.3})
BLUE, RED = "#2E5C8A", "#C0392B"

BEHAV = ["e_impr","e_clicks","e_likes","e_shares","e_comments","e_viewcomment",
         "e_homepage","e_active_days","e_avg_pos","e_click_rate","e_like_rate","e_avg_view_time"]

def add_tenure(df):
    ud = config.RAW_DIR.parent / "user_demographics.csv"   # non-leaky covariate
    if ud.exists() and "registeredMonthCnt" not in df.columns:
        df = df.join(pl.read_csv(ud).select(["userId","registeredMonthCnt"]), on="userId", how="left")
    return df


## Fit helper
Same model and shared hash split as the pipeline, so results match the chapter exactly.


In [ ]:
def fit_stage(df, target, feats):
    df = df.with_columns([pl.col(c).fill_null(0.0) for c in feats])
    assert not (set(feats) & config.FORBIDDEN_FEATURES), "leakage guard"
    def XY(s):
        d = df.filter(pl.col("split") == s)
        return d.select(feats).to_numpy(), d[target].to_numpy().astype(int)
    Xtr, ytr = XY("train"); Xte, yte = XY("test")
    clf = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.07, max_depth=6,
                                         l2_regularization=1.0, random_state=42).fit(Xtr, ytr)
    p = clf.predict_proba(Xte)[:, 1]
    return clf, Xte, yte, p, {"roc_auc": roc_auc_score(yte, p),
                              "pr_auc": average_precision_score(yte, p),
                              "base_rate": float(yte.mean())}


## Stage 1 — Retention (predict churn)


In [ ]:
ret = add_tenure(pl.read_parquet(DER / "user_retention_table.parquet"))
feats1 = BEHAV + [c for c in ["ct_video_share_seen","ct_seen_n_content","registeredMonthCnt"] if c in ret.columns]
clf1, Xte1, yte1, p1, m1 = fit_stage(ret, "churned", feats1)
print(f"users: {ret.height:,} | churn base rate {m1['base_rate']:.3f}")
print(f"ROC-AUC {m1['roc_auc']:.4f} | PR-AUC {m1['pr_auc']:.4f}")


## Stage 2 — Activeness (predict inactive, among returners)


In [ ]:
mt = add_tenure(pl.read_parquet(DER / "user_modeling_table.parquet"))
feats2 = BEHAV + [c for c in ["registeredMonthCnt"] if c in mt.columns]
clf2, Xte2, yte2, p2, m2 = fit_stage(mt, "is_inactive", feats2)
print(f"users: {mt.height:,} | inactive base rate {m2['base_rate']:.3f}")
print(f"ROC-AUC {m2['roc_auc']:.4f} | PR-AUC {m2['pr_auc']:.4f}")


## ROC and Precision–Recall curves


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for (yte, p, m, lab, col) in [(yte1, p1, m1, "Stage 1 retention", BLUE),
                              (yte2, p2, m2, "Stage 2 activeness", RED)]:
    fpr, tpr, _ = roc_curve(yte, p)
    ax[0].plot(fpr, tpr, color=col, label=f"{lab} (AUC={m['roc_auc']:.3f})")
    prec, rec, _ = precision_recall_curve(yte, p)
    ax[1].plot(rec, prec, color=col, label=f"{lab} (AP={m['pr_auc']:.3f})")
    ax[1].axhline(m["base_rate"], ls="--", color=col, alpha=0.5)
ax[0].plot([0,1],[0,1],"--",color="grey"); ax[0].set_title("ROC"); ax[0].set_xlabel("FPR"); ax[0].set_ylabel("TPR"); ax[0].legend()
ax[1].set_title("Precision–Recall"); ax[1].set_xlabel("recall"); ax[1].set_ylabel("precision"); ax[1].legend()
plt.tight_layout(); plt.show()


## Feature importance (permutation, test subsample)


In [ ]:
def imp_plot(clf, Xte, yte, feats, title, col):
    rng = np.random.default_rng(0); idx = rng.choice(len(yte), min(20000, len(yte)), replace=False)
    pi = permutation_importance(clf, Xte[idx], yte[idx], n_repeats=3, scoring="roc_auc", random_state=0, n_jobs=4)
    order = np.argsort(pi.importances_mean)
    plt.figure(figsize=(6, 4)); plt.barh([feats[i] for i in order], pi.importances_mean[order], color=col)
    plt.xlabel("ROC-AUC drop when shuffled"); plt.title(title); plt.tight_layout(); plt.show()

imp_plot(clf1, Xte1, yte1, feats1, "Stage 1 — retention", BLUE)
imp_plot(clf2, Xte2, yte2, feats2, "Stage 2 — activeness", RED)


## Funnel decomposition
P(engaged) = P(return) × P(click | return).


In [ ]:
p_return = 1 - m1["base_rate"]
p_click_given_return = 1 - m2["base_rate"]
p_engaged = p_return * p_click_given_return
print(f"P(return)              = {p_return:.3f}")
print(f"P(click | return)      = {p_click_given_return:.3f}")
print(f"P(engaged overall)     = {p_engaged:.3f}")

fig, ax = plt.subplots(figsize=(7, 3.2))
stages = ["Week-1 users", "Return to feed", "Return & click"]
vals = [1.0, p_return, p_engaged]
ax.bar(stages, vals, color=[ "#888888", BLUE, "#2E8B57"])
for i, v in enumerate(vals): ax.text(i, v + 0.02, f"{v:.0%}", ha="center")
ax.set_ylim(0, 1.1); ax.set_ylabel("share of week-1 users"); ax.set_title("Engagement funnel")
plt.tight_layout(); plt.show()


## Does tenure help? (ablation)


In [ ]:
def auc_only(df, target, feats):
    _, _, _, _, m = fit_stage(df, target, feats)
    return m["roc_auc"]

base1 = [f for f in feats1 if f != "registeredMonthCnt"]
base2 = [f for f in feats2 if f != "registeredMonthCnt"]
print(f"Stage 1 retention : without tenure {auc_only(ret,'churned',base1):.4f}  ->  with tenure {m1['roc_auc']:.4f}")
print(f"Stage 2 activeness: without tenure {auc_only(mt,'is_inactive',base2):.4f}  ->  with tenure {m2['roc_auc']:.4f}")
ten = ret.with_columns(pl.col('registeredMonthCnt').fill_null(0)).group_by('churned').agg(pl.col('registeredMonthCnt').mean().round(1)).sort('churned')
print('\nMean tenure (months) by churn:', dict(zip(ten['churned'].to_list(), ten['registeredMonthCnt'].to_list())))


## Takeaways
- Early behaviour predicts retention (ROC-AUC ~0.71) about as well as activeness (~0.71) — moderate, usable signal.
- The funnel localises loss: ~64% return, but only ~28% of returners click, so ~18% are both retained and active.
- Tenure adds a small lift and longer-tenured users churn less, but behaviour dominates how-long-registered.
- These are leakage-free reference baselines for the sequence-aware (Member B) and survival (Member C) models.
